# Preprocesamiento de Artículos, Se consideran Aquellos que tienen 130 filas

In [13]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import numpy as np
df = pd.read_csv("Base pricing NEW CSV.csv")


df_linea_blanca = df[df["Departamento"] == "VIDEO"].copy()
df_linea_blanca.drop(columns=["Inv cto", "Inv pzas", "Precio SAP"], inplace=True)
print(df_linea_blanca.columns)

renaming_dict = {
    "Año": "Year",
    "Semana": "Week",
    "Departamento": "Department",
    "Material": "Product_ID",
    "Descripción de Material": "Product_Description",
    "Marca": "Brand",
    "Grupo de Artículo": "Product_Group",
    "Venta Costo": "Sales_Cost",
    "Venta Pzs": "Units_Sold",
    "Venta": "Total_Sales"
}

df_linea_blanca.rename(columns=renaming_dict, inplace=True)

os.makedirs('split_data_VIDEO', exist_ok=True)
for product_id, group in df_linea_blanca.groupby('Product_ID'):
    filename = f"split_data_VIDEO/{product_id}.csv"
    group.to_csv(filename, index=False)
    #print(f"guardado: {filename}")

data_dir = Path(r"split_data_VIDEO") 
csv_files = list(data_dir.glob("*.csv"))
print(f"Número de archivos antes del filtro: {len(csv_files)}")

min_rows = 130
for file in data_dir.glob("*.csv"):
    try:
        df = pd.read_csv(file)
        if len(df) < min_rows:
            #print(f"Eliminando {file.name} ({len(df)} filas)")
            file.unlink()
    except Exception as e:
        print(f"Error procesando {file.name} : {e}")

csv_files = list(data_dir.glob("*.csv"))
print(f"Número de archivos después del filtro: {len(csv_files)}")

Index(['Año', 'Semana', 'Departamento', 'Material', 'Descripción de Material',
       'Marca', 'Grupo de Artículo', 'Venta Costo', 'Venta Pzs', 'Venta'],
      dtype='object')
Número de archivos antes del filtro: 252
Número de archivos después del filtro: 25


# Analisis individual de cada Artículo y Fitrado por artículos con correlación de (>.4) en Precio/Unidades

In [14]:
import os
import shutil
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import nbformat as nbf

# Directorio que contiene todos los archivos de productos
dossier = r"split_data_VIDEO"

# Nuevo directorio para productos elásticos
dossier_cible = os.path.join(dossier, "produits_elastiques")
os.makedirs(dossier_cible, exist_ok=True)

# Umbral de correlación para considerar un producto sensible
seuil_corr = 0.4

# DataFrame final para todas las métricas
df_metrics = pd.DataFrame(columns=["Produit", "Model", "RMSE", "MAE", "R2", "Price_Units_Corr"])

# Contadores para estadísticas
total_archivos = 0
archivos_elasticos = 0

# Recorrer todos los archivos Excel/CSV del directorio
for fichier in os.listdir(dossier):
    if fichier.endswith(".xlsx") or fichier.endswith(".csv"):
        total_archivos += 1
        chemin_fichier = os.path.join(dossier, fichier)
        nom_produit = os.path.splitext(fichier)[0]

        # Cargar el archivo
        df = pd.read_excel(chemin_fichier) if fichier.endswith(".xlsx") else pd.read_csv(chemin_fichier)

        # Agregar para hacer Year + Week único
        df = df.groupby(["Year", "Week"], as_index=False).agg({
            "Total_Sales": "sum",
            "Units_Sold": "sum",
            "Department": "first",
            "Product_ID": "first",
            "Product_Description": "first",
            "Brand": "first",
            "Product_Group": "first"
        })

        # Crear características temporales
        df["tendance"] = np.arange(len(df))
        df["week_sin"] = np.sin(2 * np.pi * df["Week"] / 52)
        df["week_cos"] = np.cos(2 * np.pi * df["Week"] / 52)

        # Añadir precio
        df["price"] = df["Total_Sales"] / df["Units_Sold"]
        df["price"] = df["price"].replace([np.inf, -np.inf], np.nan)
        df["price"] = df["price"].fillna(df["price"].mean())

        # Calcular correlación entre price y Units_Sold
        corr_price_units = df["price"].corr(df["Units_Sold"])

        # Verificar si es producto elástico
        if abs(corr_price_units) >= seuil_corr:
            archivos_elasticos += 1
            shutil.copy2(chemin_fichier, os.path.join(dossier_cible, fichier))
            print(f"{fichier} → corr={corr_price_units:.2f} → copiado en 'produits_elastiques'")

            # Variables explicativas y objetivo (todo el dataset)
            X = df[["price", "week_sin", "week_cos"]]
            y = df["Units_Sold"]

            # Normalizar solo el precio
            scaler = StandardScaler()
            X_scaled = X.copy()
            X_scaled[["price"]] = scaler.fit_transform(X[["price"]])

            # Definir modelos
            models = {
                "LinearRegression": LinearRegression(),
                "RandomForest": RandomForestRegressor(random_state=42),
                "XGBoost": XGBRegressor(random_state=42, eval_metric='rmse')
            }

            # Entrenar y calcular métricas en todo el dataset
            for name, model in models.items():
                model.fit(X_scaled, y)
                y_pred = model.predict(X_scaled)
                mse = mean_squared_error(y, y_pred)
                rmse = np.sqrt(mse)
                mae = mean_absolute_error(y, y_pred)
                r2 = r2_score(y, y_pred)

                df_metrics = pd.concat([df_metrics, pd.DataFrame([{
                    "Produit": nom_produit,
                    "Model": name,
                    "RMSE": rmse,
                    "MAE": mae,
                    "R2": r2,
                    "Price_Units_Corr": corr_price_units
                }])], ignore_index=True)

# Mostrar estadísticas
print(f"\nArchivos procesados: {total_archivos}")
print(f"Archivos elásticos encontrados: {archivos_elasticos}")

dossier_produits = r"./split_data_VIDEO\produits_elastiques"
dossier_final = os.path.join(dossier_produits, "produits_analyse")
os.makedirs(dossier_final, exist_ok=True)

notebooks_creados = 0

for fichier in os.listdir(dossier_produits):
    if fichier.endswith(".xlsx") or fichier.endswith(".csv"):
        nom_produit = os.path.splitext(fichier)[0]
        sous_dossier = os.path.join(dossier_final, nom_produit)
        os.makedirs(sous_dossier, exist_ok=True)
        
        shutil.copy2(os.path.join(dossier_produits, fichier), sous_dossier)
        
        nb = nbf.v4.new_notebook()
        
        texte_intro = f"# Análisis del producto {nom_produit}\n\nEste notebook aplica LinearRegression, RandomForest y XGBoost para encontrar el precio óptimo."
        nb.cells.append(nbf.v4.new_markdown_cell(texte_intro))
        
        code_cell = f"""
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Cargar datos
df = pd.read_excel("{fichier}") if "{fichier}".endswith(".xlsx") else pd.read_csv("{fichier}")

# Agregar para hacer Year + Week único
df = (
    df.groupby(["Year", "Week"], as_index=False)
    .agg({{
        "Total_Sales": "sum",
        "Units_Sold": "sum",
        "Department": "first",
        "Product_ID": "first",
        "Product_Description": "first",
        "Brand": "first",
        "Product_Group": "first"
    }})
)

# Preparar características
df["date"] = pd.to_datetime(df["Year"].astype(str) + df["Week"].astype(str) + '1', format='%Y%W%w')
df["tendance"] = np.arange(len(df))
df["week_sin"] = np.sin(2 * np.pi * df["Week"] / 52)
df["week_cos"] = np.cos(2 * np.pi * df["Week"] / 52)
df["price"] = df["Total_Sales"] / df["Units_Sold"]

df["price"] = df["price"].replace([np.inf, -np.inf], np.nan)
df["price"] = df["price"].fillna(df["price"].mean())

# Variables explicativas y objetivo (todo el dataset)
X = df[["price", "week_sin", "week_cos"]]
y = df["Units_Sold"]

# Normalización
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Definición de modelos
models = {{
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, eval_metric='rmse')
}}

results = {{}}

for name, model in models.items():
    # Entrenamiento y predicción en todo el dataset
    model.fit(X_scaled, y)
    y_pred = model.predict(X_scaled)

    # Evaluación
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    # Cálculo del precio óptimo
    prix_range = np.linspace(df["price"].min(), df["price"].max(), 50)
    revenus = []
    for p in prix_range:
        X_opt = pd.DataFrame({{
            "price": [p],
            "week_sin": [np.sin(2 * np.pi * ((df["Week"].max()+1) % 52) / 52)],
            "week_cos": [np.cos(2 * np.pi * ((df["Week"].max()+1) % 52) / 52)]
        }})
        X_opt_scaled = scaler_X.transform(X_opt)
        units_pred = model.predict(X_opt_scaled)[0]
        revenus.append(p * units_pred)
    prix_opt = prix_range[np.argmax(revenus)]

    results[name] = {{"R2": r2, "RMSE": rmse, "MAE": mae, "Prix_optimal": prix_opt}}

# Mostrar resultados
for name, res in results.items():
    print(f"=== {{name}} ===")
    print(f"R2: {{res['R2']:.3f}}, RMSE: {{res['RMSE']:.3f}}, MAE: {{res['MAE']:.3f}}, Precio óptimo: {{res['Prix_optimal']:.2f}}\\n")

# Gráfico de ingresos vs precio
plt.figure(figsize=(10,6))
for name, model in models.items():
    revenus = []
    for p in prix_range:
        X_opt = pd.DataFrame({{
            "price": [p],
            "week_sin": [np.sin(2 * np.pi * ((df["Week"].max()+1) % 52) / 52)],
            "week_cos": [np.cos(2 * np.pi * ((df["Week"].max()+1) % 52) / 52)]
        }})
        X_opt_scaled = scaler_X.transform(X_opt)
        units_pred = model.predict(X_opt_scaled)[0]
        revenus.append(p * units_pred)
    plt.plot(prix_range, revenus, label=name)

plt.xlabel("Precio")
plt.ylabel("Ingreso previsto")
plt.title("Simulación ingresos vs precio")
plt.legend()
plt.grid(True)
plt.show()
"""

        nb.cells.append(nbf.v4.new_code_cell(code_cell))
        
        chemin_nb = os.path.join(sous_dossier, f"analyse_{nom_produit}.ipynb")
        with open(chemin_nb, "w", encoding="utf-8") as f:
            nbf.write(nb, f)
        
        notebooks_creados += 1

print(f"Notebooks creados para {notebooks_creados} productos elásticos!")

2691337.csv → corr=-0.67 → copiado en 'produits_elastiques'
2691341.csv → corr=0.66 → copiado en 'produits_elastiques'


C:\Users\jose.valdez\AppData\Local\Temp\ipykernel_15788\2371481504.py:94: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, pd.DataFrame([{


2732815.csv → corr=0.77 → copiado en 'produits_elastiques'
2756901.csv → corr=0.57 → copiado en 'produits_elastiques'
2822670.csv → corr=0.45 → copiado en 'produits_elastiques'
2826976.csv → corr=-0.59 → copiado en 'produits_elastiques'
2833281.csv → corr=-0.62 → copiado en 'produits_elastiques'
2841994.csv → corr=-0.44 → copiado en 'produits_elastiques'
2854108.csv → corr=0.45 → copiado en 'produits_elastiques'

Archivos procesados: 25
Archivos elásticos encontrados: 9
Notebooks creados para 9 productos elásticos!


## Procesamiento de todos los artículos Elasticos

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Configuración inicial
dossier = r"split_data_VIDEO"
seuil_corr = 0.4  # Umbral de correlación para productos elásticos

# DataFrame para almacenar todos los resultados
resultados_finales = pd.DataFrame(columns=[
    "Product_ID", "Product_Description", "Brand", "Product_Group", 
    "Best_Model", "Best_Price", "Predicted_Units", "Expected_Revenue",
    "Model_R2", "Price_Units_Correlation", "RMSE", "MAE"
])

# Procesar todos los archivos en el directorio
for fichier in os.listdir(dossier):
    if fichier.endswith(".csv"):
        try:
            # Cargar datos
            chemin_fichier = os.path.join(dossier, fichier)
            df = pd.read_csv(chemin_fichier)
            product_id = os.path.splitext(fichier)[0]
            
            # Agregar datos por semana
            df = df.groupby(["Year", "Week"], as_index=False).agg({
                "Total_Sales": "sum",
                "Units_Sold": "sum",
                "Department": "first",
                "Product_ID": "first",
                "Product_Description": "first",
                "Brand": "first",
                "Product_Group": "first"
            })
            
            # Crear características temporales
            df["week_sin"] = np.sin(2 * np.pi * df["Week"] / 52)
            df["week_cos"] = np.cos(2 * np.pi * df["Week"] / 52)
            
            # Calcular precio y limpiar valores
            df["price"] = df["Total_Sales"] / df["Units_Sold"]
            df["price"] = df["price"].replace([np.inf, -np.inf], np.nan)
            df["price"] = df["price"].fillna(df["price"].mean())
            
            # Calcular correlación precio-unidades
            corr_price_units = df["price"].corr(df["Units_Sold"])
            
            # Solo procesar productos elásticos
            if abs(corr_price_units) >= seuil_corr:
                #print(f"Procesando {product_id} (correlación: {corr_price_units:.3f})")
                
                # Preparar datos para modelado
                X = df[["price", "week_sin", "week_cos"]]
                y = df["Units_Sold"]
                
                # Normalizar características
                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X)
                
                # Definir modelos
                models = {
                    "LinearRegression": LinearRegression(),
                    "RandomForest": RandomForestRegressor(random_state=42),
                    "XGBoost": XGBRegressor(random_state=42, eval_metric='rmse')
                }
                
                # Evaluar cada modelo
                best_model_name = ""
                best_r2 = -np.inf
                best_model = None
                best_predictions = None
                
                for name, model in models.items():
                    try:
                        # Entrenar modelo
                        model.fit(X_scaled, y)
                        y_pred = model.predict(X_scaled)
                        
                        # Evaluar modelo
                        r2 = r2_score(y, y_pred)
                        rmse = np.sqrt(mean_squared_error(y, y_pred))
                        mae = mean_absolute_error(y, y_pred)
                        
                        # Seleccionar el mejor modelo basado en R²
                        if r2 > best_r2:
                            best_r2 = r2
                            best_model_name = name
                            best_model = model
                            best_predictions = y_pred
                    except Exception as e:
                        print(f"Error con {name} para {product_id}: {str(e)}")
                        continue
                
                # Si encontramos un modelo válido, calcular precio óptimo
                if best_model is not None:
                    # Generar rango de precios para probar
                    prix_range = np.linspace(df["price"].min(), df["price"].max(), 50)
                    max_revenue = -np.inf
                    optimal_price = 0
                    predicted_units = 0
                    
                    # Encontrar precio que maximiza los ingresos
                    for p in prix_range:
                        # Preparar datos para predicción (usamos la última semana conocida)
                        X_opt = pd.DataFrame({
                            "price": [p],
                            "week_sin": [np.sin(2 * np.pi * ((df["Week"].max() + 1) % 52) / 52)],
                            "week_cos": [np.cos(2 * np.pi * ((df["Week"].max() + 1) % 52) / 52)]
                        })
                        
                        # Escalar y predecir
                        X_opt_scaled = scaler.transform(X_opt)
                        units_pred = best_model.predict(X_opt_scaled)[0]
                        revenue = p * units_pred
                        
                        # Encontrar máximo revenue
                        if revenue > max_revenue:
                            max_revenue = revenue
                            optimal_price = p
                            predicted_units = units_pred
                    
                    # Almacenar resultados
                    nueva_fila = pd.DataFrame({
                        "Product_ID": [product_id],
                        "Product_Description": [df["Product_Description"].iloc[0]],
                        "Brand": [df["Brand"].iloc[0]],
                        "Product_Group": [df["Product_Group"].iloc[0]],
                        "Best_Model": [best_model_name],
                        "Best_Price": [round(optimal_price) if optimal_price < 1000 else round(optimal_price / 100) * 100],
                        "Predicted_Units": [int(round(predicted_units))],
                        "Expected_Revenue": [round(max_revenue)],
                        "Model_R2": [best_r2],
                        "Price_Units_Correlation": [corr_price_units],
                        "RMSE": [rmse],
                        "MAE": [mae]
                    })
                    
                    resultados_finales = pd.concat([resultados_finales, nueva_fila], ignore_index=True)
                    
        except Exception as e:
            print(f"Error procesando {fichier}: {str(e)}")
            continue

# Guardar resultados finales
resultados_finales.to_csv("optimal_pricing_results.csv", index=False)
print(f"Proceso completado. Resultados guardados en optimal_pricing_results.csv")
print(f"Se analizaron {len(resultados_finales)} productos elásticos")

# Mostrar resumen de resultados
print("\nResumen de resultados:")
print(resultados_finales[["Product_ID", "Best_Model", "Best_Price", "Predicted_Units", "Expected_Revenue", "Model_R2"]])

Proceso completado. Resultados guardados en optimal_pricing_results.csv
Se analizaron 9 productos elásticos

Resumen de resultados:
  Product_ID Best_Model Best_Price Predicted_Units Expected_Revenue  Model_R2
0    2691337    XGBoost       5400             152           819147  0.999998
1    2691341    XGBoost       8200              14           116119  0.999414
2    2732815    XGBoost       3900              29           113529  1.000000
3    2756901    XGBoost       4400              36           158171  1.000000
4    2822670    XGBoost       4400               3            13918  0.999792
5    2826976    XGBoost       3600              55           200256  0.999998
6    2833281    XGBoost       2900              70           202336  1.000000
7    2841994    XGBoost      13100              14           180533  1.000000
8    2854108    XGBoost       5200              37           193013  1.000000
